# 15 — Rain Enhancement Training

Train a compact supervised rain-image enhancement model for the Adaptive_Road_Vision pipeline.

This notebook uses only **Rain** images from the training split and validation split. BDD100K does not provide paired clean/rain-free targets, so a deterministic pseudo-reference target is generated from each rainy image using conservative de-raining operations. The checkpoint is intended for later adaptive enhancement and downstream detection experiments.

**Research limitation:** the pseudo-clean target is a heuristic generated from the rainy input, not paired clean ground truth. PSNR/SSIM measure similarity to this pseudo-target only; they do not establish real deraining quality. Validate visual quality and downstream YOLO detection before claiming improvement.


In [ ]:
from pathlib import Path
import os, json, random, math, time
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter
import matplotlib.pyplot as plt

def find_project_root():
    current = Path.cwd().resolve()
    candidates = [current, *list(current.parents[:5])]
    for candidate in candidates:
        if (candidate / "datasets").exists() and (candidate / "notebooks").exists():
            return candidate
    fallback = Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision")
    if fallback.exists():
        return fallback
    return current

PROJECT_ROOT = find_project_root()

def resolve_dataset_root():
    for cand in [
        PROJECT_ROOT / "datasets" / "final_condition_balanced_v2",
        PROJECT_ROOT / "datasets" / "final_condition_balanced",
        PROJECT_ROOT / "datasets" / "final",
    ]:
        if (cand / "metadata" / "final_dataset_metadata.csv").exists():
            return cand
        if (cand / "final_dataset_metadata.csv").exists():
            return cand
        if (cand / "labels" / "train").exists() and any((cand / "labels" / "train").glob("*.txt")):
            return cand
    return PROJECT_ROOT / "datasets" / "final_condition_balanced_v2"

FINAL_DIR = resolve_dataset_root()
DATASET_ROOT = FINAL_DIR
RESULTS_DIR = PROJECT_ROOT / "results" / "enhancement"
MODEL_DIR = PROJECT_ROOT / "models" / "enhancement"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

assert PROJECT_ROOT.exists(), f"Project root not found: {PROJECT_ROOT}"
print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)
print("Results dir:", RESULTS_DIR)
print("Models dir:", MODEL_DIR)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
else:
    print("CUDA unavailable; training will use CPU.")

## 1. Configuration

Rain is much smaller than the Night subset in BDD100K, so this notebook uses all available Rain training images by default. The optional `TRAIN_LIMIT` and `VAL_LIMIT` switches are useful for quick debugging.

In [ ]:
SEED = 42
IMAGE_SIZE = 256
BATCH_SIZE = 8
NUM_WORKERS = 0
EPOCHS = 25
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-5
PATIENCE = 6
TRAIN_LIMIT = None
VAL_LIMIT = None
SAVE_EVERY = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print({
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "device": str(DEVICE)
})

In [ ]:
metadata_candidates = [
    FINAL_DIR / "metadata" / "final_dataset_metadata.csv",
    FINAL_DIR / "final_dataset_metadata.csv",
    FINAL_DIR / "metadata" / "dataset_metadata.csv",
    FINAL_DIR / "dataset_metadata.csv",
    PROJECT_ROOT / "datasets" / "processed" / "metadata" / "bdd100k_processed_metadata.csv",
]
metadata_path = next((p for p in metadata_candidates if p.is_file()), None)
if metadata_path is None and FINAL_DIR.is_dir():
    acceptable = {"final_dataset_metadata.csv", "dataset_metadata.csv", "final_metadata.csv", "balanced_dataset_metadata.csv"}
    discovered = [p for p in FINAL_DIR.rglob("*.csv") if p.name.lower() in acceptable]
    if discovered:
        metadata_path = sorted(discovered, key=lambda p: (len(p.parts), str(p).lower()))[0]
if metadata_path is None:
    raise FileNotFoundError(
        "Final dataset metadata was not found. Checked:\n" +
        "\n".join(str(p) for p in metadata_candidates) +
        "\nSet metadata_path explicitly to the CSV created by Notebook 10."
    )
meta = pd.read_csv(metadata_path)
print("Metadata:", metadata_path)
print("Shape:", meta.shape)
print("Columns:", meta.columns.tolist())
display(meta.head())

In [ ]:
def resolve_final_image(row):
    for col in ["final_image_path", "image_path", "resolved_path"]:
        if col in row.index and pd.notna(row[col]) and str(row[col]).strip():
            p = Path(str(row[col]))
            candidates = [p] if p.is_absolute() else [PROJECT_ROOT / p, metadata_path.parent / p, metadata_path.parent.parent / p]
            for candidate in candidates:
                if candidate.is_file():
                    return candidate.resolve()
    split = str(row.get("split", row.get("target_split", "train"))).strip().lower()
    split = {"validation": "val", "valid": "val", "training": "train", "testing": "test"}.get(split, split)
    name = str(row.get("image_name", row.get("image_id", row.get("filename", "")))).strip()
    if not name or name.lower() == "nan":
        return None
    if Path(name).suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
        name += ".jpg"
    candidates = [FINAL_DIR / "images" / split / name, FINAL_DIR / "images" / split / Path(name).name, FINAL_DIR / split / name]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    return None

if "split" not in meta.columns and "target_split" in meta.columns:
    meta["split"] = meta["target_split"]
if "split" not in meta.columns:
    raise ValueError("Metadata must contain 'split' or 'target_split'.")
meta["resolved_path"] = meta.apply(resolve_final_image, axis=1)
missing_path_count = int(meta["resolved_path"].isna().sum())
if missing_path_count:
    print(f"Warning: {missing_path_count} rows have unresolved image paths.")
meta = meta[meta["resolved_path"].notna()].copy()
meta["resolved_path"] = meta["resolved_path"].astype(str)
meta["split"] = meta["split"].astype(str).str.strip().str.lower()
meta["split"] = meta["split"].replace({"validation":"val", "valid":"val", "training":"train", "testing":"test"})
if "project_condition" in meta.columns:
    condition_col = "project_condition"
elif "condition" in meta.columns:
    condition_col = "condition"
    meta["project_condition"] = meta["condition"]
elif "weather" in meta.columns:
    condition_col = "weather"
    meta["project_condition"] = meta["weather"]
else:
    raise ValueError("Condition column required: project_condition, condition, or weather.")
condition_aliases = {"day":"Day","daytime":"Day","night":"Night","nighttime":"Night","rain":"Rain","rainy":"Rain","fog":"Fog","foggy":"Fog","unknown":"Unknown","unknown/other":"Unknown","unk":"Unknown"}
meta[condition_col] = meta[condition_col].astype(str).str.strip().map(lambda x: condition_aliases.get(x.lower(), x.title()))
meta["project_condition"] = meta[condition_col]
if meta["resolved_path"].duplicated().any():
    raise ValueError("Duplicate resolved image paths found in metadata.")
print("Usable rows:", len(meta))
print("Split counts:\n", meta["split"].value_counts())
print("Conditions:\n", meta[condition_col].value_counts(dropna=False))

## 2. Build rain enhancement pairs

Only **Rain** images from the training split are used for fitting. Validation uses the Rain validation split. Test images remain completely untouched.

Because paired clean images are unavailable, the pseudo-reference removes estimated rain streaks with a deterministic local smoothing/detail-preservation pipeline. This is a reproducible training target, not ground-truth restoration.

In [ ]:
rain_train = meta[
    (meta["split"].astype(str).str.lower() == "train") &
    (meta[condition_col].astype(str).str.lower() == "rain")
].copy()

rain_val = meta[
    (meta["split"].astype(str).str.lower() == "val") &
    (meta[condition_col].astype(str).str.lower() == "rain")
].copy()

if TRAIN_LIMIT is not None:
    rain_train = rain_train.sample(
        min(TRAIN_LIMIT, len(rain_train)),
        random_state=SEED
    )

if VAL_LIMIT is not None:
    rain_val = rain_val.sample(
        min(VAL_LIMIT, len(rain_val)),
        random_state=SEED
    )

print("Rain train:", len(rain_train))
print("Rain val:", len(rain_val))
print("Rain test images are intentionally excluded.")

if rain_train.empty:
    raise ValueError("No Rain images found in the training split. Check metadata labels and split names.")
if rain_val.empty:
    raise ValueError("No Rain images found in the validation split. Validation is required for checkpoint selection.")
for split_name, frame in [("train", rain_train), ("val", rain_val)]:
    missing = [p for p in frame["resolved_path"] if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError(f"{len(missing)} missing Rain {split_name} images; example: {missing[0]}")


In [ ]:
def make_reference_target(img):
    # Deterministic pseudo-clean target.
    # Mild median filtering suppresses isolated rain streaks,
    # while blending with the original image preserves scene structure.
    arr = np.asarray(img).astype(np.uint8)
    base = Image.fromarray(arr)

    smooth = base.filter(ImageFilter.MedianFilter(size=3))
    smooth = smooth.filter(ImageFilter.GaussianBlur(radius=0.35))

    a = np.asarray(base).astype(np.float32)
    b = np.asarray(smooth).astype(np.float32)

    # Conservative blend: avoid destroying road/object boundaries.
    out = 0.70 * a + 0.30 * b

    out_img = Image.fromarray(np.clip(out, 0, 255).astype(np.uint8))
    out_img = ImageEnhance.Contrast(out_img).enhance(1.03)

    return out_img


sample_img = Image.open(
    rain_train.iloc[0]["resolved_path"]
).convert("RGB")

sample_img = sample_img.resize(
    (IMAGE_SIZE, IMAGE_SIZE),
    Image.Resampling.BILINEAR
)

sample_target = make_reference_target(sample_img)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(sample_img)
axes[0].set_title("Rain source")
axes[0].axis("off")

axes[1].imshow(sample_target)
axes[1].set_title("Pseudo-reference target")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
class RainEnhancementDataset(Dataset):
    def __init__(self, frame, train=False):
        self.frame = frame.reset_index(drop=True)
        self.train = train

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        path = self.frame.iloc[idx]["resolved_path"]

        img = Image.open(path).convert("RGB")
        img = img.resize(
            (IMAGE_SIZE, IMAGE_SIZE),
            Image.Resampling.BILINEAR
        )

        target = make_reference_target(img)

        if self.train and random.random() < 0.5:
            img = TF.hflip(img)
            target = TF.hflip(target)

        x = TF.to_tensor(img)
        y = TF.to_tensor(target)

        return x, y, str(path)


train_ds = RainEnhancementDataset(rain_train, train=True)
val_ds = RainEnhancementDataset(rain_val, train=False)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

In [ ]:
# Sanity checks before training
assert len(train_ds) > 0 and len(val_ds) > 0
assert len(train_loader) > 0 and len(val_loader) > 0
print("Rain enhancement data is ready.")
print(f"Train images: {len(train_ds):,}; validation images: {len(val_ds):,}")
print("Test images are excluded from fitting and checkpoint selection.")

## 3. Rain enhancement model

A compact residual encoder-decoder predicts an RGB correction that is added to the rainy input. The architecture is deliberately similar to Notebook 14 so the later adaptive enhancement stage can use condition-specific models with consistent interfaces.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.InstanceNorm2d(channels, affine=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(channels, channels, 3, padding=1),
            nn.InstanceNorm2d(channels, affine=True),
        )

    def forward(self, x):
        return x + self.block(x)


class RainEnhancer(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(inplace=True)
        )

        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(inplace=True)
        )

        self.enc3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.ReLU(inplace=True)
        )

        self.res = nn.Sequential(
            *[ResidualBlock(128) for _ in range(4)]
        )

        self.dec2 = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1),
            nn.ReLU(inplace=True)
        )

        self.dec1 = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1),
            nn.ReLU(inplace=True)
        )

        self.out = nn.Conv2d(32, 3, 3, padding=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)

        z = self.res(e3)

        z = F.interpolate(
            z,
            size=e2.shape[-2:],
            mode="bilinear",
            align_corners=False
        )
        z = self.dec2(z) + e2

        z = F.interpolate(
            z,
            size=e1.shape[-2:],
            mode="bilinear",
            align_corners=False
        )
        z = self.dec1(z) + e1

        residual = torch.tanh(self.out(z)) * 0.5

        return torch.clamp(x + residual, 0, 1)


model = RainEnhancer().to(DEVICE)

params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Trainable parameters: {params:,}")

## 4. Losses and metrics

The objective combines L1 reconstruction, structural similarity, and a small total-variation penalty. PSNR and SSIM are reported against the pseudo-reference target.

In [ ]:
def gaussian_window(
    window_size=11,
    sigma=1.5,
    channels=3,
    device=None
):
    coords = (
        torch.arange(window_size, device=device).float()
        - window_size // 2
    )

    g = torch.exp(
        -(coords ** 2) / (2 * sigma ** 2)
    )

    g = g / g.sum()

    kernel = torch.outer(g, g)

    return kernel.expand(
        channels,
        1,
        window_size,
        window_size
    ).contiguous()


def ssim_torch(x, y, window_size=11):
    channels = x.shape[1]

    window = gaussian_window(
        window_size,
        1.5,
        channels,
        x.device
    )

    mu_x = F.conv2d(
        x,
        window,
        padding=window_size // 2,
        groups=channels
    )

    mu_y = F.conv2d(
        y,
        window,
        padding=window_size // 2,
        groups=channels
    )

    mu_x2 = mu_x * mu_x
    mu_y2 = mu_y * mu_y
    mu_xy = mu_x * mu_y

    sigma_x = (
        F.conv2d(
            x * x,
            window,
            padding=window_size // 2,
            groups=channels
        )
        - mu_x2
    )

    sigma_y = (
        F.conv2d(
            y * y,
            window,
            padding=window_size // 2,
            groups=channels
        )
        - mu_y2
    )

    sigma_xy = (
        F.conv2d(
            x * y,
            window,
            padding=window_size // 2,
            groups=channels
        )
        - mu_xy
    )

    C1 = 0.01 ** 2
    C2 = 0.03 ** 2

    score = (
        (2 * mu_xy + C1) *
        (2 * sigma_xy + C2)
        /
        (
            (mu_x2 + mu_y2 + C1) *
            (sigma_x + sigma_y + C2)
            + 1e-8
        )
    )

    return score.mean()


def total_variation(x):
    return (
        (x[:, :, :, 1:] - x[:, :, :, :-1]).abs().mean()
        +
        (x[:, :, 1:, :] - x[:, :, :-1, :]).abs().mean()
    )


def psnr_torch(x, y):
    mse = F.mse_loss(x, y).item()
    return 10 * math.log10(
        1.0 / max(mse, 1e-12)
    )


def compute_loss(pred, target):
    l1 = F.l1_loss(pred, target)
    structural = 1.0 - ssim_torch(pred, target)
    tv = total_variation(pred)

    return l1 + 0.20 * structural + 0.01 * tv

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

try:
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=torch.cuda.is_available()
    )
except Exception:
    scaler = torch.cuda.amp.GradScaler(
        enabled=torch.cuda.is_available()
    )


def amp_context():
    if torch.cuda.is_available():
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        )

    return torch.autocast(
        device_type="cpu",
        dtype=torch.bfloat16,
        enabled=False
    )

## 5. Training

The best checkpoint is selected using validation loss. Intermediate checkpoints are saved every `SAVE_EVERY` epochs.

In [ ]:
RAIN_MODEL_DIR = MODEL_DIR / "rain"
RAIN_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

history = []
best_val = float("inf")
bad_epochs = 0

best_path = RAIN_MODEL_DIR / "rain_enhancer_best.pth"
last_path = RAIN_MODEL_DIR / "rain_enhancer_last.pth"

for epoch in range(1, EPOCHS + 1):
    start = time.time()

    model.train()
    train_losses = []

    for x, y, _ in train_loader:
        x = x.to(
            DEVICE,
            non_blocking=True
        )
        y = y.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with amp_context():
            pred = model(x)
            loss = compute_loss(
                pred,
                y
            )

        if torch.cuda.is_available():
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        train_losses.append(
            loss.item()
        )

    model.eval()

    val_losses = []
    val_psnr = []
    val_ssim = []

    with torch.no_grad():
        for x, y, _ in val_loader:
            x = x.to(
                DEVICE,
                non_blocking=True
            )
            y = y.to(
                DEVICE,
                non_blocking=True
            )

            with amp_context():
                pred = model(x)
                loss = compute_loss(
                    pred,
                    y
                )

            pred_f = pred.float()
            y_f = y.float()

            val_losses.append(
                loss.item()
            )

            val_psnr.append(
                psnr_torch(
                    pred_f,
                    y_f
                )
            )

            val_ssim.append(
                ssim_torch(
                    pred_f,
                    y_f
                ).item()
            )

    train_loss = float(
        np.mean(train_losses)
    )

    val_loss = float(
        np.mean(val_losses)
    )

    mean_psnr = float(
        np.mean(val_psnr)
    )

    mean_ssim = float(
        np.mean(val_ssim)
    )

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_psnr": mean_psnr,
        "val_ssim": mean_ssim,
        "lr": optimizer.param_groups[0]["lr"],
        "seconds": time.time() - start,
    }

    history.append(row)

    scheduler.step(val_loss)

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_loss": val_loss,
        "val_psnr": mean_psnr,
        "val_ssim": mean_ssim,
        "config": {
            "image_size": IMAGE_SIZE,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "seed": SEED,
        },
    }

    torch.save(
        checkpoint,
        last_path
    )

    if epoch % SAVE_EVERY == 0:
        torch.save(
            model.state_dict(),
            RAIN_MODEL_DIR /
            f"rain_enhancer_epoch_{epoch:03d}.pth"
        )

    if val_loss < best_val:
        best_val = val_loss
        bad_epochs = 0

        torch.save(
            checkpoint,
            best_path
        )
    else:
        bad_epochs += 1

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train {train_loss:.4f} | "
        f"val {val_loss:.4f} | "
        f"PSNR {mean_psnr:.2f} | "
        f"SSIM {mean_ssim:.4f}"
    )

    if bad_epochs >= PATIENCE:
        print("Early stopping.")
        break

In [ ]:
history_df = pd.DataFrame(history)

history_path = (
    RESULTS_DIR /
    "rain_enhancement_training_history.csv"
)

history_df.to_csv(
    history_path,
    index=False
)

config = {
    "project_root": str(PROJECT_ROOT),
    "task": "rain_enhancement_training",
    "source_condition": "Rain",
    "train_count": int(len(rain_train)),
    "val_count": int(len(rain_val)),
    "test_used_for_training": False,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs_requested": EPOCHS,
    "epochs_completed": int(len(history_df)),
    "learning_rate": LEARNING_RATE,
    "device": str(DEVICE),
    "best_checkpoint": str(best_path),
    "last_checkpoint": str(last_path),
    "target_type": "deterministic_pseudo_reference",
}

with open(
    RESULTS_DIR /
    "rain_enhancement_training_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        config,
        f,
        indent=2
    )

print("Best checkpoint:", best_path)
print("Last checkpoint:", last_path)
print("History:", history_path)

## 6. Training curves

In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.plot(
    history_df["epoch"],
    history_df["train_loss"],
    label="Train loss"
)

ax.plot(
    history_df["epoch"],
    history_df["val_loss"],
    label="Val loss"
)

ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Rain enhancement training")
ax.legend()
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()


fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.plot(
    history_df["epoch"],
    history_df["val_psnr"],
    label="Val PSNR"
)

ax.set_xlabel("Epoch")
ax.set_ylabel("PSNR (dB)")
ax.set_title("Validation PSNR")
ax.legend()
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

## 7. Visual sanity check

Load the best checkpoint and compare the rainy input, model output, and pseudo-reference.

In [ ]:
checkpoint = torch.load(
    best_path,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

n_show = min(
    4,
    len(val_ds)
)

fig, axes = plt.subplots(
    n_show,
    3,
    figsize=(12, 4 * n_show)
)

if n_show == 1:
    axes = np.expand_dims(
        axes,
        axis=0
    )

for i in range(n_show):
    x, y, path = val_ds[i]

    with torch.no_grad():
        pred = model(
            x.unsqueeze(0).to(DEVICE)
        ).squeeze(0).cpu().clamp(0, 1)

    axes[i, 0].imshow(
        x.permute(1, 2, 0)
    )
    axes[i, 0].set_title(
        "Rain input"
    )
    axes[i, 0].axis("off")

    axes[i, 1].imshow(
        pred.permute(1, 2, 0)
    )
    axes[i, 1].set_title(
        "Enhanced output"
    )
    axes[i, 1].axis("off")

    axes[i, 2].imshow(
        y.permute(1, 2, 0)
    )
    axes[i, 2].set_title(
        "Pseudo-reference"
    )
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()

## 8. Final summary

Main artifact:

`models/enhancement/rain/rain_enhancer_best.pth`

Use downstream object-detection and tracking evaluation to determine whether rain enhancement improves the complete Adaptive_Road_Vision system.

**Important:** inspect enhanced outputs for blur, color shifts, and loss of small road objects. This model is trained against a heuristic pseudo-reference; use downstream YOLO metrics to measure practical value.


In [ ]:
if len(history_df):
    best_idx = history_df["val_loss"].idxmin()

    summary = {
        "best_epoch": int(
            history_df.loc[
                best_idx,
                "epoch"
            ]
        ),
        "best_val_loss": float(
            history_df.loc[
                best_idx,
                "val_loss"
            ]
        ),
        "best_val_psnr": float(
            history_df.loc[
                best_idx,
                "val_psnr"
            ]
        ),
        "best_val_ssim": float(
            history_df.loc[
                best_idx,
                "val_ssim"
            ]
        ),
        "checkpoint": str(best_path),
        "train_images": int(len(rain_train)),
        "validation_images": int(len(rain_val)),
    }
else:
    summary = {
        "error": "No training history was produced."
    }

with open(
    RESULTS_DIR /
    "rain_enhancement_training_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

print(
    json.dumps(
        summary,
        indent=2
    )
)